# F3_02_Algoritmos

    ## Objetivo

    Implementar algoritmos estructurados, recursivos, divide and conquer y exploración de grafos sobre el dataset procesado de Fase 3.

    ## Resultado esperado

    - Ranking iterativo.
    - Máximo iterativo.
    - Máximo recursivo con slicing.
    - Máximo recursivo optimizado.
    - Promedio manual por región.
    - Promedio con Pandas.
    - Búsqueda lineal y filtro por periodo.
    - Grafo de similitud regional y DFS recursivo.
    - Validaciones técnicas de casos normales, límite y excepciones.

> Nota técnica: este notebook está diseñado para ejecutarse desde cualquier subcarpeta del repositorio. 
> Las rutas se resuelven buscando automáticamente la carpeta raíz `gym-fitness-analytics`.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

## 1. Carga del dataset procesado

Se carga `data/processed/gym_data_processed.csv`.

In [2]:
def encontrar_raiz_proyecto(nombre_repo="gym-fitness-analytics"):
    ruta_actual = Path.cwd().resolve()

    for ruta in [ruta_actual] + list(ruta_actual.parents):
        if ruta.name == nombre_repo:
            return ruta

    raise FileNotFoundError(
        f"No se pudo encontrar la raíz del proyecto '{nombre_repo}' desde {ruta_actual}"
    )


PROJECT_ROOT = encontrar_raiz_proyecto()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "gym_data_processed.csv"

print("Directorio actual:", Path.cwd().resolve())
print("Raíz del proyecto:", PROJECT_ROOT)
print("Ruta dataset procesado:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset procesado: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset procesado cargado correctamente.")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

Directorio actual: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics\src\fase3
Raíz del proyecto: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics
Ruta dataset procesado: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics\data\processed\gym_data_processed.csv
Dataset procesado cargado correctamente.
Filas: 3564
Columnas: 32


,country,year,region,gym_memberships,fitness_participation_rate,total_health_club_revenue_usd,number_of_gyms,gym_penetration_rate,urban_population_percentage,obesity_rate,...,gym_penetration_rate_norm,urban_population_percentage_norm,obesity_rate_norm,gdp_per_capita_usd_norm,population_total_norm,average_membership_cost_usd_norm,insufficient_physical_activity_pct_norm,memberships_per_100k_norm,gyms_per_100k_norm,revenue_per_membership_usd_norm
0,Angola,2000,Africa,95521.0,0.3873,5731259.0,204.0,0.0059,0.5051,0.0470,...,0.012004,0.439905,0.072507,0.003299,0.011038,0.0,0.307219,0.012043,0.034128,0.000001
1,Angola,2001,Africa,103840.0,0.3939,6230372.0,222.0,0.0062,0.5172,0.0496,...,0.013004,0.453599,0.076772,0.003080,0.011419,0.0,0.302963,0.013051,0.036453,0.000001
2,Angola,2002,Africa,121093.0,0.4003,7265583.0,249.0,0.0070,0.5289,0.0522,...,0.015672,0.466840,0.081037,0.006461,0.011819,0.0,0.298550,0.015678,0.040381,0.000001
3,Angola,2003,Africa,142783.0,0.4065,8566966.0,281.0,0.0080,0.5400,0.0548,...,0.019006,0.479402,0.085302,0.007438,0.012243,0.0,0.294294,0.018908,0.044930,0.000001
4,Angola,2004,Africa,179615.0,0.4124,10776918.0,325.0,0.0097,0.5504,0.0574,...,0.024675,0.491172,0.089567,0.009747,0.012696,0.0,0.290038,0.024573,0.051321,0.000001


## 2. Preparación mínima para algoritmos

Se seleccionan columnas relevantes y se eliminan nulos críticos.

In [3]:
columnas_f3 = [
    "country",
    "region",
    "year",
    "gym_penetration_rate",
    "memberships_per_100k",
    "gyms_per_100k",
    "revenue_per_membership_usd",
    "periodo"
]

faltantes = [col for col in columnas_f3 if col not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias para F3: {faltantes}")

df_f3 = df[columnas_f3].copy()

df_f3 = df_f3.dropna(
    subset=[
        "gym_penetration_rate",
        "memberships_per_100k",
        "gyms_per_100k"
    ]
)

print("Dataset preparado para algoritmos.")
print("Filas:", df_f3.shape[0])
print("Columnas:", df_f3.shape[1])

df_f3.head()

Dataset preparado para algoritmos.
Filas: 3564
Columnas: 8


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
0,Angola,Africa,2000,0.0059,589.822616,1.259658,59.999990,pre_covid
1,Angola,Africa,2001,0.0062,620.043651,1.325594,59.999730,pre_covid
2,Angola,Africa,2002,0.0070,698.840625,1.437006,60.000025,pre_covid
3,Angola,Africa,2003,0.0080,795.727216,1.566008,59.999902,pre_covid
4,Angola,Africa,2004,0.0097,965.650082,1.747272,60.000100,pre_covid


## 3. Funciones auxiliares

Validan columnas antes de ejecutar algoritmos.

In [4]:
def validar_columna(df, columna):
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")


def validar_columnas(df, columnas):
    faltantes = [col for col in columnas if col not in df.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")

## 4. Ranking iterativo

Algoritmo estructurado con complejidad temporal `O(n log n)` por ordenamiento.

In [5]:
def calcular_ranking_paises(df, columna, top_n=10):
    if top_n <= 0:
        raise ValueError("top_n debe ser mayor que cero.")

    columnas_requeridas = ["country", "region", "year", columna]
    validar_columnas(df, columnas_requeridas)

    resultados = []

    for _, fila in df.iterrows():
        valor = fila[columna]

        if pd.notna(valor):
            resultados.append({
                "country": fila["country"],
                "region": fila["region"],
                "year": int(fila["year"]),
                "valor": float(valor)
            })

    resultados_ordenados = sorted(
        resultados,
        key=lambda x: x["valor"],
        reverse=True
    )

    return resultados_ordenados[:top_n]


df_ranking = pd.DataFrame(
    calcular_ranking_paises(
        df_f3,
        columna="memberships_per_100k",
        top_n=10
    )
)

df_ranking

,country,region,year,valor
0,United States of America,North America,2026,30221.918617
1,United States of America,North America,2025,30000.846077
2,Netherlands,Europe,2026,29194.416757
3,Norway,Europe,2023,28987.602546
4,Netherlands,Europe,2025,28980.856127
5,Netherlands,Europe,2024,28731.432516
6,Norway,Europe,2026,28597.186178
7,Norway,Europe,2025,28388.007851
8,Norway,Europe,2024,28143.691298
9,Australia,Oceania,2026,27699.213423


## 5. Máximos iterativo y recursivos

Permiten comparar programación estructurada, divide and conquer y optimización de memoria.

In [6]:
def maximo_iterativo(valores):
    if len(valores) == 0:
        return None

    maximo = valores[0]

    for valor in valores[1:]:
        if valor > maximo:
            maximo = valor

    return maximo


def maximo_recursivo(valores):
    if len(valores) == 0:
        return None

    if len(valores) == 1:
        return valores[0]

    mitad = len(valores) // 2

    max_izq = maximo_recursivo(valores[:mitad])
    max_der = maximo_recursivo(valores[mitad:])

    if max_izq is None:
        return max_der

    if max_der is None:
        return max_izq

    return max(max_izq, max_der)


def maximo_recursivo_optimizado(valores, inicio=0, fin=None):
    if fin is None:
        fin = len(valores)

    if inicio >= fin:
        return None

    if fin - inicio == 1:
        return valores[inicio]

    mitad = (inicio + fin) // 2

    max_izq = maximo_recursivo_optimizado(valores, inicio, mitad)
    max_der = maximo_recursivo_optimizado(valores, mitad, fin)

    if max_izq is None:
        return max_der

    if max_der is None:
        return max_izq

    return max(max_izq, max_der)


valores_memberships = (
    df_f3["memberships_per_100k"]
    .dropna()
    .astype(float)
    .tolist()
)

max_iter = maximo_iterativo(valores_memberships)
max_rec = maximo_recursivo(valores_memberships)
max_rec_opt = maximo_recursivo_optimizado(valores_memberships)

print("Máximo iterativo:", max_iter)
print("Máximo recursivo con slicing:", max_rec)
print("Máximo recursivo optimizado:", max_rec_opt)

assert max_iter == max_rec == max_rec_opt

print("Las tres implementaciones entregan el mismo resultado.")

Máximo iterativo: 30221.9186167546
Máximo recursivo con slicing: 30221.9186167546
Máximo recursivo optimizado: 30221.9186167546
Las tres implementaciones entregan el mismo resultado.


## 6. Promedio manual vs Pandas

Compara una implementación estructurada con una herramienta optimizada para datos tabulares.

In [7]:
def promedio_manual_por_region(df, columna):
    validar_columnas(df, ["region", columna])

    acumuladores = {}
    conteos = {}

    for _, fila in df.iterrows():
        region = fila["region"]
        valor = fila[columna]

        if pd.notna(region) and pd.notna(valor):
            region = str(region)

            if region not in acumuladores:
                acumuladores[region] = 0.0
                conteos[region] = 0

            acumuladores[region] += float(valor)
            conteos[region] += 1

    promedios = {}

    for region in acumuladores:
        promedios[region] = acumuladores[region] / conteos[region]

    return promedios


def promedio_pandas_por_region(df, columna):
    validar_columnas(df, ["region", columna])

    return df.groupby("region")[columna].mean().sort_values(ascending=False)


promedio_manual = promedio_manual_por_region(
    df_f3,
    "gym_penetration_rate"
)

promedio_pandas = promedio_pandas_por_region(
    df_f3,
    "gym_penetration_rate"
)

print("Promedio manual por región:")
display(pd.Series(promedio_manual).sort_values(ascending=False))

print("Promedio Pandas por región:")
display(promedio_pandas)

Promedio manual por región:


Europe           0.113839
Oceania          0.098022
North America    0.088760
South America    0.055014
Asia             0.050762
Africa           0.013010
dtype: float64

Promedio Pandas por región:


region
Europe           0.113839
Oceania          0.098022
North America    0.088760
South America    0.055014
Asia             0.050762
Africa           0.013010
Name: gym_penetration_rate, dtype: float64

## 7. Búsqueda lineal y filtro por periodo

Demuestran estructuras de control aplicadas al dataset.

In [8]:
def busqueda_lineal_pais(df, pais):
    validar_columna(df, "country")

    coincidencias = []

    for _, fila in df.iterrows():
        if str(fila["country"]).lower() == pais.lower():
            coincidencias.append(fila)

    if len(coincidencias) == 0:
        return pd.DataFrame(columns=df.columns)

    return pd.DataFrame(coincidencias)


def filtrar_por_periodo(df, periodo):
    validar_columna(df, "periodo")

    periodos_validos = {"pre_covid", "covid", "post_covid"}

    if periodo not in periodos_validos:
        raise ValueError(
            f"Periodo inválido: {periodo}. Use uno de: {periodos_validos}"
        )

    return df[df["periodo"] == periodo].copy()


pais_ejemplo = df_f3["country"].iloc[0]
resultado_pais = busqueda_lineal_pais(df_f3, pais_ejemplo)

print("País usado como ejemplo:", pais_ejemplo)
print("Registros encontrados:", resultado_pais.shape[0])
display(resultado_pais.head())

df_post_covid = filtrar_por_periodo(df_f3, "post_covid")
print("Registros post_covid:", df_post_covid.shape[0])
display(df_post_covid.head())

País usado como ejemplo: Angola
Registros encontrados: 27


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
0,Angola,Africa,2000,0.0059,589.822616,1.259658,59.999990,pre_covid
1,Angola,Africa,2001,0.0062,620.043651,1.325594,59.999730,pre_covid
2,Angola,Africa,2002,0.0070,698.840625,1.437006,60.000025,pre_covid
3,Angola,Africa,2003,0.0080,795.727216,1.566008,59.999902,pre_covid
4,Angola,Africa,2004,0.0097,965.650082,1.747272,60.000100,pre_covid


Registros post_covid: 660


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
22,Angola,Africa,2022,0.0304,3042.110615,4.237404,60.719986,post_covid
23,Angola,Africa,2023,0.0295,2948.978972,4.359195,59.999977,post_covid
24,Angola,Africa,2024,0.0290,2895.851694,4.384223,60.000026,post_covid
25,Angola,Africa,2025,0.0292,2920.993007,4.423815,59.999990,post_covid
26,Angola,Africa,2026,0.0294,2942.518195,4.455489,59.999975,post_covid


## 8. Grafo de similitud regional + DFS recursivo

Agrega exploración de grafos y recursividad aplicada al dataset.

In [9]:
def construir_grafo_similitud_regiones(df, columna, umbral=2.0):
    validar_columnas(df, ["region", columna])

    promedios = df.groupby("region")[columna].mean()
    regiones = list(promedios.index)

    grafo = {region: [] for region in regiones}

    for i in range(len(regiones)):
        for j in range(i + 1, len(regiones)):
            region_1 = regiones[i]
            region_2 = regiones[j]

            diferencia = abs(promedios[region_1] - promedios[region_2])

            if diferencia <= umbral:
                grafo[region_1].append(region_2)
                grafo[region_2].append(region_1)

    return grafo


def dfs_recursivo(grafo, nodo, visitados=None):
    if visitados is None:
        visitados = set()

    visitados.add(nodo)

    for vecino in grafo[nodo]:
        if vecino not in visitados:
            dfs_recursivo(grafo, vecino, visitados)

    return visitados


grafo_regiones = construir_grafo_similitud_regiones(
    df_f3,
    columna="gym_penetration_rate",
    umbral=2.0
)

display(grafo_regiones)

nodo_inicial = list(grafo_regiones.keys())[0]

regiones_visitadas = dfs_recursivo(
    grafo_regiones,
    nodo_inicial
)

print("Nodo inicial:", nodo_inicial)
print("Regiones visitadas:", regiones_visitadas)

{'Africa': ['Asia', 'Europe', 'North America', 'Oceania', 'South America'],
 'Asia': ['Africa', 'Europe', 'North America', 'Oceania', 'South America'],
 'Europe': ['Africa', 'Asia', 'North America', 'Oceania', 'South America'],
 'North America': ['Africa', 'Asia', 'Europe', 'Oceania', 'South America'],
 'Oceania': ['Africa', 'Asia', 'Europe', 'North America', 'South America'],
 'South America': ['Africa', 'Asia', 'Europe', 'North America', 'Oceania']}

Nodo inicial: Africa
Regiones visitadas: {'North America', 'Europe', 'Africa', 'South America', 'Asia', 'Oceania'}


## 9. Validaciones técnicas

Incluye casos normales, límite y excepción controlada.

In [10]:
assert maximo_iterativo([1, 5, 3]) == 5
assert maximo_recursivo([1, 5, 3]) == 5
assert maximo_recursivo_optimizado([1, 5, 3]) == 5

assert maximo_iterativo([]) is None
assert maximo_recursivo([]) is None
assert maximo_recursivo_optimizado([]) is None

assert maximo_iterativo([10]) == 10
assert maximo_recursivo([10]) == 10
assert maximo_recursivo_optimizado([10]) == 10

assert maximo_iterativo([-10, -3, -8]) == -3

print("Pruebas de casos normales y límite ejecutadas correctamente.")

try:
    calcular_ranking_paises(df_f3, "columna_inexistente")
except ValueError as error:
    print("Error controlado correctamente:", error)

Pruebas de casos normales y límite ejecutadas correctamente.
Error controlado correctamente: Faltan columnas requeridas: ['columna_inexistente']


## Conclusión

Este notebook cubre programación estructurada, recursividad, divide and conquer, exploración de grafos y validaciones técnicas.